# 13 · Frozen real-data evaluation studies

This notebook exercises the *study protocol* using **fictional OSM records and simulated labels**. It is a software test, not evidence of real-world accuracy. The real-data procedure is documented in `docs/real-data-evaluation.md`.

We will construct a synthetic OSM-derived catalog, freeze it and its queries, verify all hashes, and demonstrate why complete human review is required before evaluation.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from savoroute.data.catalog import catalog_to_dict
from savoroute.evaluation.study import (
    prepare_osm_study, verify_osm_study, evaluate_reviewed_study, StudyError,
)
from savoroute.ingestion.osm import import_osm_snapshot

root = Path.cwd()
source = root / "data/sample/osm_overpass_fixture.json"
raw = json.loads(source.read_text(encoding="utf-8"))
imported = import_osm_snapshot(raw, synthetic_fixture=True)
print("Admitted fictional restaurants:", len(imported.catalog.items))

In [ ]:
temporary = TemporaryDirectory()
work = Path(temporary.name)
catalog_path = work / "catalog.json"
catalog_path.write_text(json.dumps(catalog_to_dict(imported.catalog)), encoding="utf-8")
queries = {"query_set_version": 1, "queries": [
    {"case_id": "dev-italian", "query": "Italian restaurants", "filters": {"kind": "restaurant"}, "split": "development"},
    {"case_id": "test-mediterranean", "query": "Mediterranean restaurant", "filters": {"kind": "restaurant"}, "split": "test"},
]}
queries_path = work / "queries.json"
queries_path.write_text(json.dumps(queries), encoding="utf-8")
study_dir = work / "frozen"
manifest = prepare_osm_study(
    source_path=source, catalog_path=catalog_path, queries_path=queries_path,
    output_dir=study_dir, collected_at="2026-11-12T12:00:00Z",
    selection_protocol="Synthetic OSM fixture shipped with Savoroute",
    query_protocol="Demonstration queries written for the fictional fixture",
    allow_synthetic_fixture=True,
)
assert verify_osm_study(study_dir) == manifest
print("Frozen study:", manifest["items"], "items,", manifest["judgment_pairs"], "unlabeled pairs")

In [ ]:
reference = json.loads((study_dir / "review_reference.json").read_text())
print("One reviewer-facing evidence record (no rank/label):")
print(json.dumps(reference["records"][0], indent=2))
assert all("relevant" not in row for row in reference["records"])

## Review and adjudication boundary

In real studies, distribute separate blank reviewer files and have two people record decisions independently. For this **synthetic software demonstration only**, the next cell makes mock decisions. It must never be described as human-reviewed or real-world evidence.

In [ ]:
mock_review = json.loads((study_dir / "review_template.json").read_text())
for position, row in enumerate(mock_review["judgments"]):
    positive = position == 0
    row["reviews"] = {"reviewer_a": positive, "reviewer_b": positive}
    row["reviewer_notes"] = {
        "reviewer_a": "Simulated example" if positive else "",
        "reviewer_b": "Simulated example" if positive else "",
    }
    row["final_label"] = positive
reviewed_path = work / "mock_review.json"
reviewed_path.write_text(json.dumps(mock_review), encoding="utf-8")
dev_report = evaluate_reviewed_study(study_dir, reviewed_path, split="development")
print("Synthetic evaluation only:", json.dumps({
    "queries": dev_report["query_count"],
    "bm25_recall_at_k": dev_report["systems"]["bm25"]["recall_at_k"],
    "synthetic": dev_report["annotation_audit"]["synthetic"],
}, indent=2))

In [ ]:
try:
    evaluate_reviewed_study(study_dir, reviewed_path, split="test")
except StudyError as exc:
    print("Test access guard:", exc)

# Clean up only this temporary demonstration directory.
temporary.cleanup()

## Real study next steps

Run `savoroute osm fetch`, import its saved snapshot, prepare a **researcher-written** query set, and freeze it with `savoroute study prepare-osm` *without* `--synthetic-fixture`. The resulting reviewer files remain unlabeled until human review and explicit adjudication. Evaluate only after the judgment protocol and ranking configuration are frozen. See [real-data evaluation documentation](../docs/real-data-evaluation.md).